# 01 · Tokenizers & Chat Templates

Before a model sees text, the text becomes **integers**; before a *chat* model sees a conversation, the messages become one exact **string**. Getting either wrong silently degrades everything downstream.

1. Look inside **BPE tokenization** — how text maps to token ids, and why a leading space or capital letter changes the tokens.
2. Inspect **special tokens** and the raw **chat template** (Jinja) a model ships with.
3. Compare four template **families** — ChatML (Qwen, SmolLM2), Llama-3, and Mistral `[INST]`.
4. Reproduce each template **byte-for-byte** with a hand-written renderer (`llmlab.tokenization`) and assert equality — then show where hand-rolling breaks.

Reusable logic lives in `llmlab/tokenization.py`; tests in `tests/test_tokenization.py`. All tokenizers used here are **ungated** (no HF token needed).

In [ ]:
import pathlib
import sys

_here = pathlib.Path.cwd()
for _cand in [_here, *_here.parents]:
    if (_cand / "llmlab").is_dir():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

import pandas as pd

from llmlab import config
from llmlab import tokenization as tk

config.configure_caches()
print("device:", config.get_device())

device: mps


## 1. BPE: text → integers

A modern tokenizer splits text into **subword pieces** using byte-level BPE. Two properties surprise people:

- **Whitespace is part of the token.** `Ġ` is a visible stand-in for a leading space, glued to the *following* word — so ` Paris` and `Paris` are different tokens.
- **Everything is reversible bytes.** Unknown characters fall back to raw UTF-8 bytes, so there is no true "unknown" token.

We use SmolLM2's tokenizer (already cached from Notebook 00).

In [ ]:
tok = tk.load_tokenizer("HuggingFaceTB/SmolLM2-135M-Instruct")

text = "The transformer's KV-cache grew to 2048 tokens in Paris."
ids = tok.encode(text, add_special_tokens=False)
roundtrip = tok.decode(ids)
print(f"{len(ids)} tokens; round-trip exact: {roundtrip == text}")

pd.DataFrame([vars(r) for r in tk.token_table(tok, text)])

18 tokens; round-trip exact: True


,pos,token_id,piece,text
0,0,504,The,The
1,1,26832,Ġtransformer,transformer
2,2,506,'s,'s
3,3,659,ĠK,K
4,4,70,V,V
5,5,29,-,-
6,6,12377,cache,cache
7,7,6679,Ġgrew,grew
8,8,288,Ġto,to
9,9,216,Ġ,


### A space and a capital letter change the tokens

The same letters tokenize differently depending on context. This is why prompt formatting (stray spaces, missing newlines) is not cosmetic — it changes the integers the model actually receives, and the model was trained on one specific convention.

In [ ]:
def n_tokens(s):
    return len(tok.encode(s, add_special_tokens=False))


variants = ["Paris", " Paris", "paris", "PARIS", "2048", "20 48", "🤖", "ofthe", " of the"]
pd.DataFrame(
    [
        (repr(v), n_tokens(v), tok.convert_ids_to_tokens(tok.encode(v, add_special_tokens=False)))
        for v in variants
    ],
    columns=["text", "n_tokens", "pieces"],
)

,text,n_tokens,pieces
0,'Paris',1,[Paris]
1,' Paris',1,[ĠParis]
2,'paris',2,"[par, is]"
3,'PARIS',2,"[PAR, IS]"
4,'2048',4,"[2, 0, 4, 8]"
5,'20 48',5,"[2, 0, Ġ, 4, 8]"
6,'🤖',3,"[ðŁ, ¤, ĸ]"
7,'ofthe',2,"[oft, he]"
8,' of the',2,"[Ġof, Ġthe]"


## 2. Special tokens & the raw chat template

Chat models add **control tokens** (begin/end of turn, role markers) and ship a **chat template** — a small Jinja program that turns a list of messages into the exact training-time string. Let's look at both for SmolLM2.

In [ ]:
print("vocab size      :", tok.vocab_size)
print("special tokens  :", tok.special_tokens_map)
print("bos / eos / pad :", repr(tok.bos_token), repr(tok.eos_token), repr(tok.pad_token))
print("\n--- chat_template (Jinja source) ---\n")
print(tok.chat_template)

vocab size      : 49152
special tokens  : {'bos_token': '<|im_start|>', 'eos_token': '<|im_end|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|im_end|>'}
bos / eos / pad : '<|im_start|>' '<|im_end|>' '<|im_end|>'

--- chat_template (Jinja source) ---

{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
' }}{% endif %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


## 3. Four template families

Different model families wrap the *same* conversation in very different control tokens. We render one fixed conversation through each family's **official** template (all four are ungated; tokenizers cache after the first download).

- **ChatML** (Qwen, SmolLM2, OpenHermes): `<|im_start|>` / `<|im_end|>` around each role.
- **Llama-3** (Llama-3.x): `<|start_header_id|>role<|end_header_id|>` headers; turns end with `<|eot_id|>`.
- **Mistral [INST]** (Mistral / Mixtral Instruct): `[INST] … [/INST]`, with **no system role** — system text is folded into the first user turn.

In [ ]:
conversation = [
    {"role": "system", "content": "You are concise."},
    {"role": "user", "content": "Name the capital of France."},
]

families = [
    ("ChatML · SmolLM2", "HuggingFaceTB/SmolLM2-135M-Instruct"),
    ("ChatML · Qwen2.5", "Qwen/Qwen2.5-0.5B-Instruct"),
    ("Llama-3", "NousResearch/Meta-Llama-3-8B-Instruct"),
    ("Mistral [INST]", "mistralai/Mistral-7B-Instruct-v0.3"),
]

rendered = {}
rows = []
for label, model_id in families:
    t = tk.load_tokenizer(model_id)
    try:
        s = t.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    except Exception:
        s = t.apply_chat_template(conversation, tokenize=False)
    rendered[label] = (t, s)
    print(f"### {label}\n{s!r}\n")
    rows.append((label, len(t.encode(s, add_special_tokens=False)), s.replace(chr(10), "\\n")[:64] + "…"))

pd.DataFrame(rows, columns=["family", "n_tokens", "preview"])

### ChatML · SmolLM2
'<|im_start|>system\nYou are concise.<|im_end|>\n<|im_start|>user\nName the capital of France.<|im_end|>\n<|im_start|>assistant\n'



### ChatML · Qwen2.5
'<|im_start|>system\nYou are concise.<|im_end|>\n<|im_start|>user\nName the capital of France.<|im_end|>\n<|im_start|>assistant\n'



### Llama-3
'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are concise.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nName the capital of France.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'



### Mistral [INST]
'<s>[INST] You are concise.\n\nName the capital of France.[/INST]'



,family,n_tokens,preview
0,ChatML · SmolLM2,24,<|im_start|>system\nYou are concise.<|im_end|>...
1,ChatML · Qwen2.5,23,<|im_start|>system\nYou are concise.<|im_end|>...
2,Llama-3,25,<|begin_of_text|><|start_header_id|>system<|en...
3,Mistral [INST],16,<s>[INST] You are concise.\n\nName the capital...


## 4. Reproduce the template byte-for-byte

The payoff: rebuild each official string with a hand-written renderer and assert they are **identical**. If we truly understand a template, we can reproduce it exactly — including the default system prompts that Qwen and SmolLM2 silently inject when you don't pass one.

In [ ]:
checks = [
    ("ChatML · SmolLM2", tk.render_chatml, {"default_system": tk.SMOLLM2_DEFAULT_SYSTEM}),
    ("ChatML · Qwen2.5", tk.render_chatml, {"default_system": tk.QWEN_DEFAULT_SYSTEM}),
    ("Llama-3", tk.render_llama3, {}),
    ("Mistral [INST]", tk.render_mistral_inst, {}),
]

results = []
for label, renderer, kwargs in checks:
    t, _ = rendered[label]
    matches, official, ours = tk.verify_render(t, conversation, renderer, **kwargs)
    print(f"{'MATCH ' if matches else 'DIFFER'}  {label}")
    if not matches:
        print(tk.first_diff(official, ours))
    results.append({"family": label, "byte_exact": matches})

assert all(r["byte_exact"] for r in results), "a renderer drifted from the official template"
print("\nAll renderers reproduce the official templates byte-for-byte ✓")

MATCH   ChatML · SmolLM2
MATCH   ChatML · Qwen2.5
MATCH   Llama-3
MATCH   Mistral [INST]

All renderers reproduce the official templates byte-for-byte ✓


### Two gotchas

**Double-BOS.** A rendered template often *already contains* the begin-of-text token. If you then tokenize it with `add_special_tokens=True` (the default), you get a **duplicated** BOS — a subtle bug that degrades quality. Tokenize pre-rendered prompts with `add_special_tokens=False`.

**Dynamic templates.** Some templates aren't pure string concatenation. Llama-3.1 injects today's date via `strftime_now(...)`, and many inject tool-call blocks. That is exactly why you call the official template in real code and keep hand-rolled renderers (like ours) for *understanding*, not production.

In [ ]:
import json

llama_tok, llama_str = rendered["Llama-3"]
bos_id = llama_tok.bos_token_id
with_special = llama_tok.encode(llama_str, add_special_tokens=True)
without_special = llama_tok.encode(llama_str, add_special_tokens=False)
print("BOS id:", bos_id)
print(f"add_special_tokens=True  → first ids {with_special[:3]} | BOS count {with_special.count(bos_id)}")
print(f"add_special_tokens=False → first ids {without_special[:3]} | BOS count {without_special.count(bos_id)}")

out = {
    "families_checked": [r["family"] for r in results],
    "all_byte_exact": bool(all(r["byte_exact"] for r in results)),
    "double_bos_with_default_tokenization": with_special.count(bos_id) > without_special.count(bos_id),
}
path = config.RESULTS_DIR / "01_tokenizers_and_chat_templates.json"
path.write_text(json.dumps(out, indent=2))
print("\nwrote", path)
print(json.dumps(out, indent=2))

BOS id: 128000
add_special_tokens=True  → first ids [128000, 128000, 128006] | BOS count 2
add_special_tokens=False → first ids [128000, 128006, 9125] | BOS count 1

wrote /Users/iali/workplace/projects/personal/ml-notes/docs/LLM/results/01_tokenizers_and_chat_templates.json
{
  "families_checked": [
    "ChatML \u00b7 SmolLM2",
    "ChatML \u00b7 Qwen2.5",
    "Llama-3",
    "Mistral [INST]"
  ],
  "all_byte_exact": true,
  "double_bos_with_default_tokenization": true
}
